In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import gsw
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import seawater as sw
from itertools import product

In [ ]:
from argo_interp.data import data_filter, get_data
from argo_interp.cycle.adapter import PchipAdapter
from argo_interp.cycle.model import Model
from argo_interp.cycle.domain import ModelData, ModelMeta
from argo_interp.cycle.config import ModelSettings, ModelKwargs

In [ ]:
data_path = Path('../../../data')
data_path.mkdir(exist_ok=True)

data_file = data_path / 'argo_data.pkl'

In [ ]:
override = False

if data_file.exists() and not override:
    with data_file.open('br') as f:
        ds = pickle.load(f)
else:
    box = [
        80, 99, ## Longitude min/max
        6, 23, ## Latitude min/max
        0, 750, ## Pressure min/max
        '2011-01-01', '2020-12-31', ## Datetime min/max
    ]
    ds = get_data(box, progress=True)

    with data_file.open('bw') as f:
        pickle.dump(ds, f)

In [ ]:
settings = ModelSettings(n_folds=2)

In [ ]:
models = {}

cycles = len(ds[['PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION']].to_dataframe().drop_duplicates())
t = tqdm(ds.groupby(['PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION']), total=cycles)
for (platform_number, cycle_number, direction), cycle_ds in t:
    pressure = cycle_ds['PRES'].values
    temperature = cycle_ds['TEMP'].values
    salinity = cycle_ds['PSAL'].values

    latitude = cycle_ds['LATITUDE'].values[0]
    longitude = cycle_ds['LONGITUDE'].values[0]
    timestamp = cycle_ds['TIME'].values[0]

    if cycle_ds.sizes['N_POINTS'] < 2:
        continue

    cycle_id = f"{int(platform_number)}-{int(cycle_number)}-{direction}"
    # Duplicate pressure rows are averaged so each cycle becomes one profile function.
    model_data = ModelData(
        pressure=pressure,
        temperature=temperature,
        salinity=salinity,
    ).clean_duplicates('mean')

    model_meta = ModelMeta(
        cycle_id=cycle_id,
        latitude=latitude,
        longitude=longitude,
        timestamp=timestamp,
        profile_pressure=(pressure.min(), pressure.max()),
    )

    model = Model.build(model_meta, model_data, PchipAdapter, settings)
    models[cycle_id] = model
    t.set_postfix(model_count=len(models))